In [ ]:
# Image Loader for Convolutional Neural Network

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import torch
torch.load('/content/drive/MyDrive/isl_model_2.pth',map_location="cpu")

OrderedDict([('convolution.0.weight',
              tensor([[[[-1.6434e-01,  4.7802e-02,  1.9933e-01],
                        [ 2.5638e-02, -1.1805e-01,  1.7398e-01],
                        [ 7.2073e-02,  1.4929e-01, -1.2921e-01]],
              
                       [[ 9.5524e-02, -1.3897e-01,  2.2458e-01],
                        [-5.8864e-02,  8.6392e-02,  7.5291e-02],
                        [ 4.3781e-02,  1.2322e-02,  4.9212e-02]],
              
                       [[-8.4648e-02, -1.8550e-01, -1.6275e-02],
                        [ 1.5179e-01, -6.0504e-02,  9.3461e-02],
                        [-6.7068e-02,  1.6406e-01,  8.0030e-02]]],
              
              
                      [[[ 9.7170e-02, -5.6152e-02, -2.3016e-01],
                        [ 4.3329e-02, -6.0327e-02,  9.8556e-02],
                        [-1.4563e-01,  1.1149e-01,  1.7244e-01]],
              
                       [[ 1.2532e-01,  1.0602e-01, -2.5773e-01],
                        [ 2.1932e-01,

In [8]:
from torchvision import transforms
import json

from PIL import Image

!wget https://raw.githubusercontent.com/yashaswi1306/Indian_Sign_Language_RealTime_Recognition/main/class_names.json

with open('class_names.json','r') as f:
  class_names=json.load(f)

--2026-06-28 12:57:50--  https://raw.githubusercontent.com/yashaswi1306/Indian_Sign_Language_RealTime_Recognition/main/class_names.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 135 [text/plain]
Saving to: ‘class_names.json.2’

class_names.json.2  100%[===================>]     135  --.-KB/s    in 0s      

2026-06-28 12:57:50 (3.09 MB/s) - ‘class_names.json.2’ saved [135/135]



In [14]:
!wget https://raw.githubusercontent.com/yashaswi1306/Indian_Sign_Language_RealTime_Recognition/main/model_isl.py

--2026-06-28 13:06:22--  https://raw.githubusercontent.com/yashaswi1306/Indian_Sign_Language_RealTime_Recognition/main/model_isl.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1168 (1.1K) [text/plain]
Saving to: ‘model_isl.py’

model_isl.py        100%[===================>]   1.14K  --.-KB/s    in 0s      

2026-06-28 13:06:22 (64.3 MB/s) - ‘model_isl.py’ saved [1168/1168]



In [18]:
from model_isl import ISL_CNN # DONT USE .PY IN THIS!

In [19]:
model_0=ISL_CNN(len(class_names))
model_0.load_state_dict(torch.load('/content/drive/MyDrive/isl_model_2.pth',map_location="cpu"))
model_0.eval()

ISL_CNN(
  (convolution): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (fully_connected): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=32768, out_features=512, bias=True)
    (2): ReLU()
    (3):

In [20]:
transform_img=transforms.Compose(
   [
    transforms.Resize((128,128)),
    transforms.ToTensor(), # pixels lie between 0 and 1
    transforms.Normalize(mean=[0.5,0.5,0.5],std=[0.5,0.5,0.5]) # pixels lie btw -1 -> 1. [0.5,0.5,0.5] for [R,G,B]
    ]
    )

In [24]:
def image_eval(img):
  img=img.convert('RGB')
  tensor_img=transform_img(img) # [3,128,128]
  tensor_img=tensor_img.unsqueeze(dim=0) # so it becomes batch, colour,h,width so [1,3,128,128]
  with torch.inference_mode():
    preds=model_0(tensor_img)
    pred_idx=preds.argmax(dim=1).item()
    preds=preds.softmax(dim=1) # so u get probabilities of each class
    prob=preds[0][pred_idx].item() # prob of predicted class
    return class_names[pred_idx],prob